# fsQCA 组态分析 + 人口学(生命表 / 分解)

这本 notebook 走两条**社会科学里 Python 长期空白**的方法链,共用同一根
`StudyState` 脊柱(state-centric,契约 `requires → produces`):

1. **模糊集定性比较分析(fsQCA)** — Ragin 的集合论方法:把「条件的**组态**」
   映射到结果。我们让 `sv.tl.qca` 从模糊隶属度里**复原**一个已知的集合关系
   `Y ⇐ (A AND B) OR C`,全程走真值表 → 一致性/覆盖率 → Quine–McCluskey
   布尔最小化,拿回最简充分性解。
2. **形式人口学** — `sv.tl.life_table` 把年龄别死亡率 `mx` 逐列构造成周期
   生命表(`qx→lx→ndx→nLx→Tx→ex`),给出各年龄预期寿命(含出生时 `e0`);
   `sv.tl.decomposition` 用 **Kitagawa(1955)** 把两人群的**粗死亡率之差**
   加法拆成「率效应 + 年龄构成效应」,并附一个 Oaxaca–Blinder 回归分解伴侣。

## 涉及函数
- `sv.tl.qca(state, conditions=, outcome=, threshold=, ...)`
  — 真值表 + 一致性/覆盖率 + Quine–McCluskey 最小化(conservative/complex 解)
- `sv.tl.life_table(state, age=, mx=, width=)` — 周期生命表 → `e0` 与各年龄 `ex`
- `sv.tl.decomposition(state, ...)` — Kitagawa 粗率差分解(+ Oaxaca–Blinder)

## 对标现实冠军包
| 方法 | R / Python 冠军 | Python 现状 |
|---|---|---|
| fsQCA | R `QCA`(Adrian Duşa)、`SetMethods` | **空白**(`fuzzy-qca`/`pyqca` 单薄失维) |
| 生命表 / 分解 | R `demography`、`DemoDecomp`;`oaxaca` | **空白**(散在脚本,无 registry) |

socialverse 的差异不在「又实现了一遍算法」,而在**注册表 grounding**:每个函数
声明 `requires`(缺就抛 `RegistryError`,不猜)与 `produces`(写回哪个 slot),
于是整条分析链自带可追溯的 provenance。下面每步都先讲**为什么 + 契约**,再跑真实输出。

In [1]:
# 让本 worktree 的 socialverse 包优先于任何已安装版本被导入,并把工作目录切到
# 本 notebook 所在目录,使 fig_*.png 与 notebook 同目录(教学用,可安全删除)。
import os
import sys

_NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_PKG_ROOT = os.path.dirname(_NB_DIR)  # gap-methods worktree 根
if _PKG_ROOT not in sys.path:
    sys.path.insert(0, _PKG_ROOT)
os.chdir(_NB_DIR)

import socialverse as sv
from socialverse import datasets as ds

print("socialverse from:", os.path.dirname(sv.__file__))
print("socialverse version:", getattr(sv, "__version__", "(dev)"))

socialverse from: /Users/fernandozeng/Desktop/analysis/omicos-project/socialverse-worktrees/gap-methods/socialverse
socialverse version: 0.1.0


---
## 第一部分 · fsQCA:从模糊隶属度复原 `Y ⇐ (A AND B) OR C`

### 数据
`ds.load_qca()` 造了 40 个案例,四列都是 `[0,1]` 的**模糊集隶属度**:
条件 `A, B, C` 和结果 `Y`,其中 `Y = max(min(A,B), C)` 再加一点噪声 —
也就是**真实的集合关系** `(A 且 B) 或 C`。QCA 的活儿就是**不看生成公式**、
只从数据把这个最简充分性解找回来。

In [2]:
qca_df = ds.load_qca()
print("shape:", qca_df.shape)
print(qca_df.head(6).to_string(index=False))

shape: (40, 5)
case    A    B    C    Y
  c1 0.64 0.57 0.76 0.80
  c2 0.27 0.32 0.50 0.56
  c3 0.04 0.59 0.53 0.57
  c4 0.02 0.34 0.79 0.83
  c5 0.81 0.39 0.41 0.41
  c6 0.91 0.89 0.73 0.82


### 契约:先看 `qca` 要什么(requires)

`sv.tl.qca` 注册时声明了 `requires={"sources": ["datasets"], "variables": ["outcome"]}`。
这是**注册表 grounding** 的关键:契约不满足就**抛 `RegistryError`,而不是
猜一个默认值**。我们故意先在一个**空 state** 上调用,把这条护栏演示出来。

In [3]:
st_bad = sv.StudyState()  # 什么都没写,requires 两项都缺
try:
    sv.tl.qca(st_bad, conditions=["A", "B", "C"], outcome="Y", threshold=0.5)
except sv.RegistryError as e:
    print("RegistryError(符合预期):")
    print(e)

RegistryError(符合预期):
socialverse.tl._setmethods.qca cannot run — unmet requires:
  - sources.datasets (produced by: ingest)
  - variables.outcome (user-supplied input)
Query registry.get_prerequisites(...) or registry.resolve_plan(...) to plan the chain.


### 满足契约后再跑

把结果变量名写进 `variables.outcome`,把数据集挂进 `sources.datasets`,
契约就满足了。`qca` 会:

1. 校准 → 每列作为 `[0,1]` 模糊集;
2. 枚举 `2^k` 个组态角(corner),每个角算**一致性**(充分性
   `Σmin(X,Y)/ΣX`)、PRI 一致性、案例数;
3. 一致性 ≥ `threshold` 且 PRI 达标的角编码为「充分路径」;
4. **Quine–McCluskey** 把这些角约简成素蕴涵,贪心取本质覆盖 → 最简 sum-of-products;
5. 在模糊数据上重算**解一致性 / 解覆盖率**。

这里把 `threshold` 设为 `0.5`(阈值偏低,让含噪的充分角进得来),看能否复原原关系。

In [4]:
st = sv.StudyState()
st.write("variables", "outcome", "Y")
st.write("sources", "datasets", ds.load_qca())

sv.tl.qca(st, conditions=["A", "B", "C"], outcome="Y", threshold=0.5)

qca_model = st.models["qca"]
print("解表达式 solution :", qca_model["solution"])
print("解一致性 consistency:", qca_model["solution_consistency"])
print("解覆盖率 coverage   :", qca_model["solution_coverage"])
print("解类型            :", qca_model["solution_type"])
print("估计器            :", qca_model["estimator"])

解表达式 solution : C + A*B
解一致性 consistency: 0.9722
解覆盖率 coverage   : 0.9765
解类型            : conservative/complex (无 remainder 简化假设)
估计器            : fsQCA_truthtable + Quine-McCluskey


**读出**:解表达式应当把 `(A 且 B) 或 C` 复原出来(等价写法,如 `C + A*B`)。
这就是「复原了已知参数/关系」这条 socialverse 主张的实证 —— 不是套公式,
是从模糊数据端到端把生成机制找回来了。

下面看每条**路径**的 raw 一致性 / raw 覆盖率,以及整张**真值表**(按一致性排序)。

In [5]:
print("=== 各充分路径(solution paths) ===")
for p in qca_model["paths"]:
    print(f"  {p['term']:>10} | raw consistency={p['raw_consistency']:.3f}"
          f" | raw coverage={p['raw_coverage']:.3f}")

import pandas as pd

tt = pd.DataFrame(st.diagnostics["consistency_coverage"]["truth_table"])
print("\n=== 真值表(前 8 行,按一致性降序;outcome=1 即入选充分角) ===")
print(tt.head(8).to_string(index=False))

=== 各充分路径(solution paths) ===
           C | raw consistency=0.981 | raw coverage=0.841
         A*B | raw consistency=0.975 | raw coverage=0.560

=== 真值表(前 8 行,按一致性降序;outcome=1 即入选充分角) ===
configuration  n  consistency    pri  outcome
        A*B*C 10       0.9941 0.9899        1
       A*~B*C  4       0.9927 0.9839        1
       ~A*B*C  3       0.9908 0.9748        1
      ~A*~B*C  8       0.9822 0.9654        1
       A*B*~C  3       0.9699 0.8920        1
      ~A*B*~C  1       0.9465 0.4595        0
     ~A*~B*~C  3       0.8377 0.2316        0
      A*~B*~C  6       0.8075 0.1667        0


### 画图:组态一致性 vs 案例数

每个点是一个组态角:横轴案例数、纵轴充分性一致性,红点是被编码为
`outcome=1` 的充分路径。红色虚线是 `threshold` 一致性阈值。

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ["#d62728" if r == 1 else "#7f7f7f" for r in tt["outcome"]]
ax.scatter(tt["n"], tt["consistency"], c=colors, s=90, edgecolor="black",
           linewidth=0.6, zorder=3)
for _, r in tt.iterrows():
    ax.annotate(r["configuration"], (r["n"], r["consistency"]),
                fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(0.5, color="#d62728", ls="--", lw=1, label="一致性阈值 threshold=0.5")
ax.set_xlabel("案例数 n(> 0.5 隶属该组态)")
ax.set_ylabel("充分性一致性 consistency = Σmin(X,Y)/ΣX")
ax.set_title("fsQCA 真值表:各组态角的一致性与案例数")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig("fig_qca_truthtable.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved fig_qca_truthtable.png")

saved fig_qca_truthtable.png


/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/844775355.py:17: UserWarning: Glyph 26696 (\N{CJK UNIFIED IDEOGRAPH-6848}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/844775355.py:17: UserWarning: Glyph 20363 (\N{CJK UNIFIED IDEOGRAPH-4F8B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/844775355.py:17: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/844775355.py:17: UserWarning: Glyph 38582 (\N{CJK UNIFIED IDEOGRAPH-96B6}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/844775355.py:17: UserWarning: Glyph 23646 (\N{CJK UNIFIED IDEOGRAPH-5C5E}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s50

![fsQCA 真值表](fig_qca_truthtable.png)

---
## 第二部分 · 形式人口学:生命表 `e0` + Kitagawa 分解

### 数据
`ds.load_demography()` 给两个人群 A、B 的**年龄别死亡率** `mx` 和**人口暴露**
`pop`(9 个年龄组,含不等宽区间 `n_years`)。设计上 **B 各年龄死亡率更高、
且人口更老**;两条链要分别回答:

- 生命表:A 群的**出生时预期寿命 `e0`** 是多少?各年龄 `ex` 如何递减?
- 分解:B 与 A 的**粗死亡率之差**,有多少来自「死得更凶(率效应)」、
  多少来自「人更老(年龄构成效应)」?

In [7]:
demo_df = ds.load_demography()
print(demo_df.to_string(index=False))

age_group  n_years   mx_A    mx_B  pop_A  pop_B
        0        1 0.0060 0.00780   12.0    8.0
      1-4        4 0.0004 0.00048   45.0   30.0
     5-14       10 0.0002 0.00022  120.0   95.0
    15-24       10 0.0009 0.00126  130.0  110.0
    25-44       20 0.0018 0.00270  260.0  230.0
    45-64       20 0.0080 0.01040  220.0  240.0
    65-74       10 0.0250 0.03000   90.0  130.0
    75-84       10 0.0700 0.07700   55.0   90.0
      85+       15 0.1800 0.18900   25.0   55.0


### 生命表:`life_table`

契约:`requires={"sources": ["datasets"]}` → `produces={"models": ["life_table"]}`。
内部按 Preston-Heuveline-Guillot 的标准列算法逐列构造:
`mx → ax → qx → px → lx → ndx → nLx → Tx → ex`,婴儿区间 `a0≈0.1`,
开区间 `a=1/m`。我们先对**人群 A** 建表。

In [8]:
st_lt = sv.StudyState()
st_lt.write("sources", "datasets", ds.load_demography())

sv.tl.life_table(st_lt, age="age_group", mx="mx_A", width="n_years")

lt = st_lt.models["life_table"]
print("出生时预期寿命 e0 (人群 A):", round(lt["e0"], 2), "岁")
print("\n=== 生命表(人群 A)===")
print(lt["table"].round(4).to_string(index=False))

出生时预期寿命 e0 (人群 A): 75.06 岁

=== 生命表(人群 A)===
  age    n     mx      ax     qx          lx        ndx          nLx           Tx      ex
    0  1.0 0.0060  0.1000 0.0060 100000.0000   596.7774   99462.9003 7505652.1306 75.0565
  1-4  4.0 0.0004  2.0000 0.0016  99403.2226   158.9180  397295.0543 7406189.2303 74.5065
 5-14 10.0 0.0002  5.0000 0.0020  99244.3046   198.2903  991451.5942 7008894.1759 70.6226
15-24 10.0 0.0009  5.0000 0.0090  99046.0143   887.4207  986023.0389 6017442.5818 60.7540
25-44 20.0 0.0018 10.0000 0.0354  98158.5935  3471.2273 1928459.5977 5031419.5429 51.2581
45-64 20.0 0.0080 10.0000 0.1481  94687.3662 14027.7580 1753469.7453 3102959.9452 32.7706
65-74 10.0 0.0250  5.0000 0.2222  80659.6083 17924.3574  716974.2959 1349490.1999 16.7307
75-84 10.0 0.0700  5.0000 0.5185  62735.2509 32529.3893  464705.5621  632515.9040 10.0823
  85+ 15.0 0.1800  5.5556 1.0000  30205.8615 30205.8615  167810.3419  167810.3419  5.5556


再对**人群 B**(死亡率整体更高)建表,直接对比两群的各年龄 `ex`,
预期 B 的 `e0` 更低。

In [9]:
st_lt.write("sources", "datasets", ds.load_demography())
sv.tl.life_table(st_lt, age="age_group", mx="mx_B", width="n_years")
lt_B = st_lt.models["life_table"]
print("出生时预期寿命 e0 (人群 B):", round(lt_B["e0"], 2), "岁")
print("A - B 的 e0 差 :", round(lt["e0"] - lt_B["e0"], 2), "岁(A 更长寿)")

出生时预期寿命 e0 (人群 B): 72.25 岁
A - B 的 e0 差 : 2.8 岁(A 更长寿)


### 画图:两群各年龄预期寿命 `ex`

In [10]:
tb_A = lt["table"]
ex_B = list(lt_B["ex"].values())

fig, ax = plt.subplots(figsize=(7, 4.5))
xpos = range(len(tb_A))
ax.plot(xpos, tb_A["ex"], "-o", color="#1f77b4", label=f"人群 A (e0={lt['e0']:.1f})")
ax.plot(xpos, ex_B, "-s", color="#d62728", label=f"人群 B (e0={lt_B['e0']:.1f})")
ax.set_xticks(list(xpos))
ax.set_xticklabels(tb_A["age"], rotation=45, ha="right", fontsize=8)
ax.set_xlabel("年龄组 age group")
ax.set_ylabel("预期寿命 ex(年)")
ax.set_title("周期生命表:各年龄预期寿命 ex(A vs B)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("fig_lifetable_ex.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved fig_lifetable_ex.png")

saved fig_lifetable_ex.png


/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/1432750168.py:15: UserWarning: Glyph 24180 (\N{CJK UNIFIED IDEOGRAPH-5E74}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/1432750168.py:15: UserWarning: Glyph 40836 (\N{CJK UNIFIED IDEOGRAPH-9F84}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/1432750168.py:15: UserWarning: Glyph 32452 (\N{CJK UNIFIED IDEOGRAPH-7EC4}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/1432750168.py:15: UserWarning: Glyph 39044 (\N{CJK UNIFIED IDEOGRAPH-9884}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/1432750168.py:15: UserWarning: Glyph 26399 (\N{CJK UNIFIED IDEOGRAPH-671F}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3

![生命表 ex](fig_lifetable_ex.png)

### Kitagawa 分解:`decomposition`

契约:`requires={"sources": ["datasets"]}` →
`produces={"models": ["decomposition"], "diagnostics": ["components"]}`。

把 `crude_B − crude_A` 加法拆成两项(记 `c` 为年龄构成份额 `pop/Σpop`):

- **率效应** `Σ (mB − mA)·(cA+cB)/2` —— 各年龄死亡率不同带来的差;
- **构成效应** `Σ (cB − cA)·(mA+mB)/2` —— 年龄结构不同带来的差。

两项**精确相加**等于总差(adding-up 残差 ≈ 0),这是 Kitagawa 分解的定义性质。

In [11]:
st_dec = sv.StudyState()
st_dec.write("sources", "datasets", ds.load_demography())

sv.tl.decomposition(st_dec)

dec = st_dec.models["decomposition"]
print("方法             :", dec["method"])
print("crude_A (粗死亡率):", round(dec["crude_A"], 5))
print("crude_B (粗死亡率):", round(dec["crude_B"], 5))
print("总差 total_diff   :", round(dec["total_diff"], 5))
print("  率效应   rate_effect        :", round(dec["rate_effect"], 5))
print("  构成效应 composition_effect :", round(dec["composition_effect"], 5))
print("adding-up 残差(应≈0)        :", round(dec["adding_up_residual"], 12))
if "oaxaca" in dec:
    ox = dec["oaxaca"]
    print("Oaxaca-Blinder 伴侣: endowments=%.5f, coefficients=%.5f"
          % (ox["endowments"], ox["coefficients"]))

方法             : Kitagawa
crude_A (粗死亡率): 0.01365
crude_B (粗死亡率): 0.02488
总差 total_diff   : 0.01123
  率效应   rate_effect        : 0.00231
  构成效应 composition_effect : 0.00892
adding-up 残差(应≈0)        : 0.0
Oaxaca-Blinder 伴侣: endowments=0.00579, coefficients=0.00544


### 画图:总差 = 率效应 + 构成效应

一张瀑布式条形:总差被拆成两块,验证「两效应相加 = 总差」的 adding-up 性质。

In [12]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
labels = ["率效应\nrate", "构成效应\ncomposition", "总差\ntotal"]
vals = [dec["rate_effect"], dec["composition_effect"], dec["total_diff"]]
bars = ax.bar(labels, vals,
              color=["#1f77b4", "#ff7f0e", "#2ca02c"], edgecolor="black")
for b, v in zip(bars, vals):
    ax.annotate(f"{v:.4f}", (b.get_x() + b.get_width() / 2, v),
                ha="center", va="bottom" if v >= 0 else "top", fontsize=9)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("对粗死亡率差的贡献")
ax.set_title("Kitagawa 分解:crude_B − crude_A = 率效应 + 构成效应")
fig.tight_layout()
fig.savefig("fig_kitagawa.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved fig_kitagawa.png")

saved fig_kitagawa.png


/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/443059102.py:12: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/443059102.py:12: UserWarning: Glyph 25928 (\N{CJK UNIFIED IDEOGRAPH-6548}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/443059102.py:12: UserWarning: Glyph 24212 (\N{CJK UNIFIED IDEOGRAPH-5E94}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/443059102.py:12: UserWarning: Glyph 26500 (\N{CJK UNIFIED IDEOGRAPH-6784}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s503s9r616083n7w440000gn/T/ipykernel_37831/443059102.py:12: UserWarning: Glyph 25104 (\N{CJK UNIFIED IDEOGRAPH-6210}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/4m/2xw3_2s50

![Kitagawa 分解](fig_kitagawa.png)

### 逐年龄贡献(diagnostics.components)

分解不只给总量,还留下**每个年龄组**的率贡献与构成贡献,便于定位差异来自哪段年龄。

In [13]:
comp = st_dec.diagnostics["components"]
print(comp.round(6).to_string(index=False))

  age   mx_A    mx_B  share_A  share_B  rate_contribution  composition_contribution
    0 0.0060 0.00780 0.012539 0.008097           0.000019                 -0.000031
  1-4 0.0004 0.00048 0.047022 0.030364           0.000003                 -0.000007
 5-14 0.0002 0.00022 0.125392 0.096154           0.000002                 -0.000006
15-24 0.0009 0.00126 0.135841 0.111336           0.000044                 -0.000026
25-44 0.0018 0.00270 0.271682 0.232794           0.000227                 -0.000087
45-64 0.0080 0.01040 0.229885 0.242915           0.000567                  0.000120
65-74 0.0250 0.03000 0.094044 0.131579           0.000564                  0.001032
75-84 0.0700 0.07700 0.057471 0.091093           0.000520                  0.002471
  85+ 0.1800 0.18900 0.026123 0.055668           0.000368                  0.005451


---
## Provenance:整条链的可追溯记录

两段分析用了各自独立的 `StudyState`。下面打印 QCA 链与人口学链的 `summary()`,
每个 slot 里挂了什么、跑了几步都在案 —— 这就是**注册表 grounding** 的产物:
契约驱动的、可复现的分析轨迹。

In [14]:
print("=== QCA 链 state ===")
print(st.summary())
print("\n=== 生命表链 state ===")
print(st_lt.summary())
print("\n=== 分解链 state ===")
print(st_dec.summary())

=== QCA 链 state ===
StudyState
  sources: datasets
  variables: outcome
  models: qca
  diagnostics: consistency_coverage
  provenance: 1 step(s)

=== 生命表链 state ===
StudyState
  sources: datasets
  models: life_table
  provenance: 2 step(s)

=== 分解链 state ===
StudyState
  sources: datasets
  models: decomposition
  diagnostics: components
  provenance: 1 step(s)


### socialverse 的差异,一句话

这两条方法链(fsQCA 组态分析、形式人口学生命表/分解)在 Python 生态里长期
**只有 R 冠军包**(`QCA`/`SetMethods`、`demography`/`DemoDecomp`)。socialverse
把它们落到同一根 `StudyState` 脊柱上,靠**注册表 grounding**(`requires` 缺就
抛 `RegistryError`「查而非猜」、`produces` 写回明确 slot)保证每步契约明确、
全链可追溯;而且用**已知生成机制的合成数据**验证了方法本身 —— fsQCA 从模糊隶属度
**复原了 `(A AND B) OR C`**,Kitagawa 分解满足**两效应精确相加**的定义性质。